# Alim'confiance : recherche et carte des contrôles sanitaires

Ce notebook interroge la base Alim'confiance (DGAL) pour retrouver les établissements alimentaires par ville, niveau d'hygiène ou type d'activité, puis affiche les résultats sur une carte interactive.

La base contient plus de 76 000 établissements avec leur dernier résultat de contrôle sanitaire et leurs coordonnées GPS.

## 1. Importation des données

On récupère les données directement depuis l'API OpenDataSoft de la DGAL, sans passer par un téléchargement manuel du fichier.

Le jeu de données a connu quelques interruptions de service par le passé. La cellule ci-dessous essaie donc deux points d'accès différents avant d'abandonner, et si aucun ne répond, on pourra charger un fichier CSV téléchargé à la main depuis le portail (voir la remarque à la fin de la cellule).

In [1]:
import pandas as pd
import numpy as np
import requests
import io
import folium
import geopandas as gpd
from shapely.geometry import Point
import ipywidgets as widgets
from IPython.display import display, clear_output

pd.set_option("display.max_colwidth", 80)

In [2]:
def charger_donnees(chemin_local=None):
    """
    Charge la base Alim'confiance.

    Si chemin_local est fourni, on lit directement ce fichier CSV
    (utile si les deux points d'accès de l'API sont indisponibles).
    Sinon on essaie successivement deux URLs de l'API DGAL.
    """
    if chemin_local:
        return pd.read_csv(chemin_local, sep=";", low_memory=False)

    sources = [
        "https://dgal.opendatasoft.com/api/explore/v2.1/catalog/datasets/export_alimconfiance/exports/csv?use_labels=true",
        "https://dgal.opendatasoft.com/explore/dataset/export_alimconfiance/download/?format=csv&use_labels_for_header=true",
    ]

    for url in sources:
        try:
            reponse = requests.get(url, timeout=60)
            reponse.raise_for_status()
            # les exports OpenDataSoft utilisent le point-virgule comme séparateur
            df = pd.read_csv(io.StringIO(reponse.text), sep=";", low_memory=False)
            if len(df) > 0:
                print(f"Données chargées depuis {url}")
                print(f"{len(df)} lignes récupérées")
                return df
        except Exception as e:
            print(f"Échec avec {url} ({e})")

    raise RuntimeError(
        "Impossible de joindre l'API. Téléchargez le CSV depuis le portail "
        "et rappelez charger_donnees(chemin_local='votre_fichier.csv')."
    )

df_brut = charger_donnees()
df_brut.head(3)

Données chargées depuis https://dgal.opendatasoft.com/api/explore/v2.1/catalog/datasets/export_alimconfiance/exports/csv?use_labels=true
71040 lignes récupérées


,APP_Libelle_activite_etablissement,APP_Libelle_etablissement,Code_postal,com_name,Synthese_eval_sanit,Date_inspection,geores,Agrement,Contexte_inspection,reg_name,...,geo_ban,geo_mobile,lieu_intervention,lieu_intervention_adresse,lieu_intervention_codepostal,lieu_intervention_ville,ods_type_activite,reg_code,dep_code,com_code
0,Rayon fromagerie,SUPER U,48100,Marvejols,Satisfaisant,2026-07-09T00:00:00+00:00,"44.551785, 3.29264",NaN,Programmation,Occitanie,...,"44.551785, 3.29264",NaN,NaN,NaN,NaN,NaN,Autres,76.0,48,48092
1,Restaurants,LE CHAMPLAIN,14600,Honfleur,A améliorer,2026-02-23T00:00:00+00:00,"49.421505, 0.233296",NaN,Programmation,Normandie,...,"49.421505, 0.233296",NaN,NaN,NaN,NaN,NaN,Autres,28.0,14,14333
2,Producteurs fermiers,C' DE CHEZ NOUS,23000,Guéret,Satisfaisant,2026-03-06T00:00:00+00:00,"46.168605, 1.870612",NaN,Programmation,Nouvelle-Aquitaine,...,"46.168605, 1.870612",NaN,NaN,NaN,NaN,NaN,Autres,75.0,23,23096


## 2. Nettoyage et préparation

Les noms de colonnes varient légèrement selon la version de l'API (majuscules ou minuscules), donc on commence par repérer automatiquement les colonnes utiles plutôt que de fixer leurs noms en dur.

On en profite aussi pour séparer les coordonnées GPS, qui arrivent dans une seule colonne au format `latitude, longitude`.

In [3]:
def trouver_colonne(colonnes, candidats):
    """Retourne le premier nom de colonne existant parmi une liste de candidats possibles."""
    colonnes_minuscules = {c.lower(): c for c in colonnes}
    for candidat in candidats:
        if candidat.lower() in colonnes_minuscules:
            return colonnes_minuscules[candidat.lower()]
    return None

colonnes = df_brut.columns.tolist()

col_nom = trouver_colonne(colonnes, ["APP_Libelle_etablissement", "nom_etablissement"])
col_activite = trouver_colonne(colonnes, ["APP_Libelle_activite_etablissement", "activite"])
col_niveau = trouver_colonne(colonnes, ["Synthese_eval_sanit", "synthese_eval_sanit"])
col_geo = trouver_colonne(colonnes, ["geores", "geo_point_2d"])
col_commune = trouver_colonne(colonnes, ["Libelle_commune", "com_name", "commune"])
col_date = trouver_colonne(colonnes, ["Date_inspection", "date_inspection"])
col_adresse = trouver_colonne(colonnes, ["Adresse_2_UA", "adresse"])

print("Nom :", col_nom)
print("Activité :", col_activite)
print("Niveau :", col_niveau)
print("Géo :", col_geo)
print("Commune :", col_commune)
print("Date :", col_date)

Nom : APP_Libelle_etablissement
Activité : APP_Libelle_activite_etablissement
Niveau : Synthese_eval_sanit
Géo : geores
Commune : Libelle_commune
Date : Date_inspection


In [4]:
def extraire_lat_lon(valeur):
    """Sépare une valeur du type '48.117, -1.677' en deux nombres."""
    if pd.isna(valeur):
        return (np.nan, np.nan)
    try:
        lat_str, lon_str = str(valeur).split(",")
        return (float(lat_str.strip()), float(lon_str.strip()))
    except ValueError:
        return (np.nan, np.nan)

df = pd.DataFrame({
    "etablissement": df_brut[col_nom],
    "activite": df_brut[col_activite],
    "niveau": df_brut[col_niveau],
    "commune": df_brut[col_commune],
    "adresse": df_brut[col_adresse] if col_adresse else "",
    "date_inspection": df_brut[col_date] if col_date else "",
})

coordonnees = df_brut[col_geo].apply(extraire_lat_lon)
df["latitude"] = coordonnees.apply(lambda x: x[0])
df["longitude"] = coordonnees.apply(lambda x: x[1])

avant = len(df)
df = df.dropna(subset=["latitude", "longitude"])
print(f"{avant - len(df)} lignes sans coordonnées ont été écartées, {len(df)} lignes conservées")

df.head(3)

1743 lignes sans coordonnées ont été écartées, 69297 lignes conservées


,etablissement,activite,niveau,commune,adresse,date_inspection,latitude,longitude
0,SUPER U,Rayon fromagerie,Satisfaisant,Marvejols,Place des Cordeliers,2026-07-09T00:00:00+00:00,44.551785,3.292640
1,LE CHAMPLAIN,Restaurants,A améliorer,Honfleur,6 PL HAMELIN,2026-02-23T00:00:00+00:00,49.421505,0.233296
2,C' DE CHEZ NOUS,Producteurs fermiers,Satisfaisant,Guéret,place bonnyaud,2026-03-06T00:00:00+00:00,46.168605,1.870612


## 3. Les quatre niveaux d'hygiène

Alim'confiance distingue quatre niveaux officiels, chacun associé à une couleur dans l'application mobile. On garde le même code couleur ici :

* Très satisfaisant : vert
* Satisfaisant : bleu
* À améliorer : orange
* À corriger de manière urgente : rouge

In [5]:
COULEURS_NIVEAU = {
    "Très satisfaisant": "green",
    "Satisfaisant": "blue",
    "À améliorer": "orange",
    "A améliorer": "orange",
    "À corriger de manière urgente": "red",
    "A corriger de manière urgente": "red",
}

EMOJIS_NIVEAU = {
    "Très satisfaisant": "🟢",
    "Satisfaisant": "🔵",
    "À améliorer": "🟠",
    "A améliorer": "🟠",
    "À corriger de manière urgente": "🔴",
    "A corriger de manière urgente": "🔴",
}

df["niveau"].value_counts()

,count
niveau,
Satisfaisant,48344
Très satisfaisant,16643
A améliorer,3981
A corriger de manière urgente,329


## 4. Fonction de recherche

La fonction `rechercher` filtre la base par ville, niveau d'hygiène et type d'établissement. Les trois paramètres sont facultatifs et se combinent entre eux.

In [6]:
def rechercher(ville=None, niveau=None, activite=None):
    resultat = df.copy()

    if ville:
        resultat = resultat[resultat["commune"].str.contains(ville, case=False, na=False)]

    if niveau:
        resultat = resultat[resultat["niveau"].str.contains(niveau, case=False, na=False)]

    if activite:
        resultat = resultat[resultat["activite"].str.contains(activite, case=False, na=False)]

    return resultat

exemple = rechercher(ville="Rennes", niveau="Satisfaisant")
print(f"{len(exemple)} établissements trouvés")
exemple.head()

200 établissements trouvés


,etablissement,activite,niveau,commune,adresse,date_inspection,latitude,longitude
756,SCARABEE,Rayon boucherie-charcuterie,Satisfaisant,Rennes,"132, rue Eugène Pottier",2026-02-26T00:00:00+00:00,48.102233,-1.713248
763,SCARABEE,"Rayon fromagerie,Rayon traiteur,Rayon pain/viennoiserie/pâtisserie",Satisfaisant,Rennes,"132, rue Eugène Pottier",2026-02-26T00:00:00+00:00,48.102233,-1.713248
1101,L'INCROYABLE DARDILLAC,Purification/Expédition de coquillages,Très satisfaisant,Marennes-Hiers-Brouage,RUE DES MARTYRS,2026-09-07T00:00:00+00:00,45.791651,-1.123874
1659,MC DONALDS,Restaurants,Très satisfaisant,Marennes-Hiers-Brouage,AV DE LATTRE DE TASSIGNY,2026-03-19T00:00:00+00:00,45.828604,-1.116335
1667,FLEUR DE RIZ,Restaurants,Satisfaisant,Rennes,46 RUE D ANTRAIN,2026-02-27T00:00:00+00:00,48.140456,-1.659343


## 5. Tableau lisible

Une petite fonction pour afficher uniquement les colonnes utiles, avec des noms plus parlants que ceux du fichier source.

In [7]:
def afficher_tableau(resultat):
    tableau = resultat[["etablissement", "activite", "niveau", "latitude", "longitude"]].copy()
    tableau.columns = ["Établissement", "Activité", "Niveau", "Latitude", "Longitude"]
    return tableau.reset_index(drop=True)

afficher_tableau(exemple)

,Établissement,Activité,Niveau,Latitude,Longitude
0,SCARABEE,Rayon boucherie-charcuterie,Satisfaisant,48.102233,-1.713248
1,SCARABEE,"Rayon fromagerie,Rayon traiteur,Rayon pain/viennoiserie/pâtisserie",Satisfaisant,48.102233,-1.713248
2,L'INCROYABLE DARDILLAC,Purification/Expédition de coquillages,Très satisfaisant,45.791651,-1.123874
3,MC DONALDS,Restaurants,Très satisfaisant,45.828604,-1.116335
4,FLEUR DE RIZ,Restaurants,Satisfaisant,48.140456,-1.659343
...,...,...,...,...,...
195,TACOZH,Restaurants,Satisfaisant,48.110465,-1.678847
196,MAXIME CORNUEJOLS,Boucherie-Charcuterie,Satisfaisant,48.909094,1.430721
197,CORNER SHELTER,Restaurants,Satisfaisant,48.110296,-1.684477
198,INTERMARCHE,Rayon pain/viennoiserie/pâtisserie,Satisfaisant,48.115486,-1.680799


## 6. Conversion en GeoDataFrame

GeoPandas attend une géométrie plutôt que deux colonnes séparées. On construit donc un point à partir de la latitude et de la longitude pour chaque ligne.

In [8]:
def vers_geodataframe(resultat):
    geometrie = [Point(lon, lat) for lon, lat in zip(resultat["longitude"], resultat["latitude"])]
    return gpd.GeoDataFrame(resultat, geometry=geometrie, crs="EPSG:4326")

geo_exemple = vers_geodataframe(exemple)
geo_exemple.head(3)

,etablissement,activite,niveau,commune,adresse,date_inspection,latitude,longitude,geometry
756,SCARABEE,Rayon boucherie-charcuterie,Satisfaisant,Rennes,"132, rue Eugène Pottier",2026-02-26T00:00:00+00:00,48.102233,-1.713248,POINT (-1.71325 48.10223)
763,SCARABEE,"Rayon fromagerie,Rayon traiteur,Rayon pain/viennoiserie/pâtisserie",Satisfaisant,Rennes,"132, rue Eugène Pottier",2026-02-26T00:00:00+00:00,48.102233,-1.713248,POINT (-1.71325 48.10223)
1101,L'INCROYABLE DARDILLAC,Purification/Expédition de coquillages,Très satisfaisant,Marennes-Hiers-Brouage,RUE DES MARTYRS,2026-09-07T00:00:00+00:00,45.791651,-1.123874,POINT (-1.12387 45.79165)


## 7. Carte interactive avec Folium

Chaque établissement devient un marqueur coloré selon son niveau d'hygiène. La carte se centre automatiquement sur la moyenne des points affichés, et retombe sur le centre de la France si la recherche ne renvoie aucun résultat.

In [9]:
CENTRE_FRANCE = (46.6, 2.5)

def creer_carte(resultat):
    if len(resultat) == 0:
        centre = CENTRE_FRANCE
        zoom = 6
    else:
        centre = (resultat["latitude"].mean(), resultat["longitude"].mean())
        zoom = 12

    carte = folium.Map(location=centre, zoom_start=zoom, tiles="CartoDB positron")

    for _, ligne in resultat.iterrows():
        couleur = COULEURS_NIVEAU.get(ligne["niveau"], "gray")
        popup_html = f"""
        <b>{ligne['etablissement']}</b><br>
        {ligne["activite"]}<br>
        Niveau : {ligne['niveau']}<br>
        {ligne['adresse']}
        """
        folium.CircleMarker(
            location=(ligne["latitude"], ligne["longitude"]),
            radius=6,
            color=couleur,
            fill=True,
            fill_color=couleur,
            fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=250),
        ).add_to(carte)

    return carte

creer_carte(exemple)

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


## 8. Filtres interactifs

Cette dernière partie ajoute des menus déroulants au dessus d'une carte et d'un tableau qui se mettent à jour au clic sur le bouton. Les valeurs de la liste "Activité" sont construites à partir des activités réellement présentes dans la base.

In [10]:
liste_niveaux = ["Tous"] + sorted(df["niveau"].dropna().unique().tolist())
liste_activites = ["Tous"] + sorted(df["activite"].dropna().unique().tolist())

widget_ville = widgets.Text(description="Ville :", placeholder="ex. Rennes")
widget_niveau = widgets.Dropdown(description="Niveau :", options=liste_niveaux, value="Tous")
widget_activite = widgets.Dropdown(description="Activité :", options=liste_activites, value="Tous")
bouton_recherche = widgets.Button(description="Rechercher", button_style="success")
zone_resultat = widgets.Output()

def on_click(_):
    with zone_resultat:
        clear_output(wait=True)
        ville = widget_ville.value or None
        niveau = None if widget_niveau.value == "Tous" else widget_niveau.value
        activite = None if widget_activite.value == "Tous" else widget_activite.value

        resultat = rechercher(ville=ville, niveau=niveau, activite=activite)
        print(f"{len(resultat)} établissements trouvés")
        display(afficher_tableau(resultat).head(20))
        display(creer_carte(resultat))

bouton_recherche.on_click(on_click)

display(widgets.VBox([widget_ville, widget_niveau, widget_activite, bouton_recherche, zone_resultat]))

## Pour aller plus loin

Quelques pistes si vous voulez prolonger ce notebook : ajouter un filtre par département à partir du code postal, exporter les résultats d'une recherche en CSV, ou grouper les marqueurs avec `folium.plugins.MarkerCluster` quand une recherche renvoie beaucoup d'établissements.